In [0]:
# 1. 声明原始文件在 Volume 里的路径
products_csv_path = "/Volumes/workspace/default/olist_files/olist_products_dataset.csv"
translation_csv_path = "/Volumes/workspace/default/olist_files/product_category_name_translation.csv"

In [0]:
# 2. 分布式读取 CSV 文件
# .option("inferSchema", "true") 会让 Spark 分布式扫描一遍数据，自动判定字段是 String 还是 Integer
raw_products_df = spark.read.format("csv") \
    .option("header", "True") \
    .option("inferSchema", "True") \
    .load(products_csv_path)

raw_translation_df = spark.read.format("csv") \
    .option("header", "True") \
    .option("inferSchema", "True") \
    .load(translation_csv_path)

In [0]:
raw_products_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_length: integer (nullable = true)
 |-- product_description_length: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [0]:
display(raw_products_df.limit(5))

product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13


In [0]:
raw_products_df.show(5)

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_length|product_description_length|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|                 40|                       287|                 1|             225|               16|               10|              14|
|3aa071139cb16b67c...|                artes|                 44|                       276|                 1|            1000|               30|               18|              20|
|96bd76ec8810374ed...|        esporte_lazer|                 46|                       250|    

In [0]:
total_products = raw_products_df.count()
print(f"There are {total_products:,} products in the raw_products_df")

There are 32,951 products in the raw_products_df


In [0]:
# 1. 挑选需要的核心字段
cropped_df = raw_products_df.select(
    "product_id", 
    "product_category_name", 
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
)

# 看看裁剪后的表骨架，多余的字段是不是全消失了
cropped_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [0]:
from pyspark.sql import functions as F

In [0]:
# 过滤掉类目为空的脏数据
clean_df = cropped_df.filter(F.col("product_category_name").isNotNull())

# 比对干净表和脏表的行数，看看过滤了多少脏数据
total_clean_products = clean_df.count()
total_dirty_products = cropped_df.count() - total_clean_products
print(f"There are {total_clean_products:,} products in the clean_df")
print(f"There are {total_dirty_products:,} products in the cropped_df")

There are 32,341 products in the clean_df
There are 610 products in the cropped_df


In [0]:
# 3. 链式连招：原地覆盖重量，并衍生体积新列
transformed_df = clean_df \
    .withColumn("product_weight_g", F.col("product_weight_g") / 1000.0) \
    .withColumn("product_volume_cm3", F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm"))

# 验证结果：我们甚至可以在 select 里直接用 .alias() 顺便把列名改得更专业
final_preview_df = transformed_df.select(
    "product_id",
    "product_category_name",
    F.col("product_weight_g").alias("product_weight_kg"), # 改名为 kg 表达更精准
    "product_volume_cm3"
)

display(final_preview_df.limit(5))

product_id,product_category_name,product_weight_kg,product_volume_cm3
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,0.225,2240
3aa071139cb16b67ca9e5dea641aaa2f,artes,1.0,10800
96bd76ec8810374ed1b65e291975717f,esporte_lazer,0.154,2430
cef67bcfe19066a932b7673e239eb23d,bebes,0.371,2704
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,0.625,4420


In [0]:
# studying aggregation functions

final_preview_df = final_preview_df.groupBy("product_category_name").agg(
    F.avg("product_weight_kg").alias("avg_weight_kg"),
    F.max("product_volume_cm3").alias("max_volume_cm3"),
    F.min("product_volume_cm3").alias("min_volume_cm3"),
    F.count("product_id").alias("total_products")
)
display(final_preview_df.limit(5))

product_category_name,avg_weight_kg,max_volume_cm3,min_volume_cm3,total_products
ferramentas_jardim,3.1037768924302784,198476,352,753
papelaria,1.7631130742049472,115200,352,849
moveis_sala,8.934846153846152,294000,1600,156
musica,1.2135185185185184,56400,352,27
livros_importados,0.596774193548387,5625,1200,31
